# DNS tunnelling detection from Zeek `dns.log` (BiLSTM + multi-head attention)

This notebook trains the project's deep-learning detector on the GraphTunnel captures
([DNS-Tunnel-Datasets](https://github.com/ggyggy666/DNS-Tunnel-Datasets),
[paper](https://ieeexplore.ieee.org/document/10636232)). The features come from Zeek `dns.log` files, the
same logs the live pipeline (`capture_live.py` → `run_zeek.py`) writes, instead of from parsing PCAPs, so the
trained model can later score live traffic.

**Data flow:** download → `Data/raw/GraphTunnel` (deduplicated layout) → `Ardashes_scripts/process_pcaps.py`
(Zeek) → `Ardashes_scripts/reorder_and_rezeek.py` (time-sort out-of-order PCAPs) → `zeek_feature_extraction`
→ `dataset_splits` (`Data/processed/GraphTunnel/splits.csv`) → this notebook → `models/zeek_bilstm/` and
`results/zeek_run.md`. See `DATA_PIPELINE.md`.

**Categories:** `normal` (benign lookups of the Cloudflare top-1M list), `tunnel` (DNS-shell, dnscat2, dnspot,
iodine, tuns), `wildcard` (benign traffic that looks structurally like tunnelling), `unknownTunnel` (tools never
used for training), and `crossEndPoint` (iodine on Android: an unseen platform, not an unseen tool).

In [ ]:
import os
import time
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt

# Project modules; they sit next to this notebook.
import dataset_splits as ds
import model_artifacts as ma
import zeek_experiments as zx
import zeek_feature_extraction as zfe

STARTED_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Run 2 retrains config B and the leave-one-family-out folds with the new default
# feature set (without the artefact-suspect pair) and an epoch cap of 150.
# Run 1 (all 30 features, 50 epochs) is the first section of results/zeek_run.md;
# its models are in models/zeek_bilstm/<name>/.
RUN_NAME = "run2_default_150ep"   # models: models/zeek_bilstm/<RUN_NAME>/<name>/, results: results/runs/<RUN_NAME>/
RUN_TITLE = "Run 2: default features without the artefact-suspect pair, epoch cap 150"
REFERENCE_RUN = "run1_all_50ep"   # shown next to this run in the report (None: no comparison)
RUN_CONFIGS = ['B']               # main configurations; the full sweep is ['B', 'A']
RUN_ABLATIONS = False             # True: also train config B with every other feature set
RUN_LOFO_FAMILIES = list(ds.TUNNEL_FAMILIES)  # leave-one-tunnel-family-out folds
RENDER_REPORT = True              # write this run's section of results/zeek_run.md
N_OF_MODELS = 5                   # training runs per configuration; min / max / avg are reported
REBUILD_FEATURES = False          # True rebuilds dataset_zeek/ from the Zeek logs even if the cache is current
print(f"run {RUN_NAME} | default feature set {zfe.DEFAULT_FEATURE_SET} ({len(zfe.FEATURE_COLUMNS)} features) | "
      f"window {zfe.WINDOW_SECONDS} s | train/val row caps per window {ds.ROW_CAPS} | sampling seed {ds.SAMPLING_SEED}")

## Part 1: features from Zeek `dns.log`

`zeek_experiments.load_features` builds one row per DNS record for every capture in
`Data/processed/GraphTunnel/capture_manifest.csv` (plus finished live sessions from `datas/zeek/`, category
`own_benign`, once there are any), with:

* **lexical** features of the query name (length, labels, character mix, entropy) and the qtype,
* **response** features (answer count, minimum TTL, rcode),
* **per-domain, per-window** aggregates over the same base domain in the same 60 s window: volume (query count,
  unique names, rate, duration, unique-subdomain ratio), shape (name length and entropy, TXT/NULL share, qtype
  diversity) and "blackhole" ratios (NXDOMAIN, no response, rejected, rcode entropy).

Every row then gets its split from `splits.csv` in both configurations:

| category | rule |
|---|---|
| normal | file index 00000–00047 train, 00048–00054 val, 00055–00061 test, 00062–00067 held out |
| tunnel | per capture: first ~70% of its windows train, next ~15% val, last ~15% test, one unused window in between |
| wildcard | **config B (primary):** the capture with the most windows among 00000–00006 is val, the other six train, 00007–00012 held out. **Config A (stress test):** all held out |
| unknownTunnel / crossEndPoint | held out only |

Train and val rows are sampled per (capture, window), at most 200 rows (1,000 for wildcard), with a fixed seed.
The aggregates were computed on all rows first, and test and held-out rows are always scored in full.
The table is cached in `dataset_zeek/` (gitignored). The old PCAP-based cache in `dataset/` is left alone.

In [ ]:
t0 = time.time()
features = zx.load_features(rebuild=REBUILD_FEATURES, n_jobs=8)
print(f"{len(features):,} rows in {time.time() - t0:.0f} s")
capture_stats = pd.DataFrame(features.attrs["capture_stats"])
print(capture_stats.groupby("category")[["records", "rejoined", "unmatched_responses_dropped", "malformed_lines"]].sum())

In [ ]:
split_counts = {}
for config in ds.CONFIGS:
    assignment = features[[f"split_{config}", f"role_{config}"]].set_axis(["split", "role"], axis=1)
    split_counts[config] = ds.split_counts(features, assignment, features[f"sample_{config}"])
    print(f"\nConfig {config} ({ds.CONFIG_DESCRIPTIONS[config]})")
    print(split_counts[config].to_string(index=False))

### Quick sanity check: do the extracted features separate the classes? (config B training rows)

In [ ]:
train_rows = features[zx.split_masks(features, ds.PRIMARY_CONFIG)["train"]]
summary_cols = ['qname_len', 'entropy', 'digit_ratio', 'hex_ratio',
                'domain_query_count', 'domain_unique_subdomain_ratio',
                'domain_query_rate', 'response_min_ttl']
print(train_rows.groupby('Label')[summary_cols].mean().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, color in zip(['benign', 'tunnel'], ['#4374B3', '#FF0B04']):
    subset = train_rows[train_rows['Label'] == label]
    axes[0].hist(subset['qname_len'], bins=40, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['entropy'], bins=40, alpha=0.6, label=label, color=color)
axes[0].set_title('Query name length'); axes[0].set_xlabel('characters'); axes[0].legend()
axes[1].set_title('Query name entropy'); axes[1].set_xlabel('bits/char'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# The PCAP-based feature cache (dataset/) is kept for comparison when it exists.
old_files = ['dataset/train.csv', 'dataset/val.csv', 'dataset/test.csv', 'dataset/heldout.csv']
if all(os.path.exists(f) for f in old_files):
    old_features = pd.concat([pd.read_csv(f) for f in old_files], ignore_index=True)
    compare_cols = [c for c in summary_cols if c in old_features.columns]
    print("PCAP-based features (dataset/), mean per class:")
    print(old_features.groupby('Label')[compare_cols].mean().T)
else:
    print("No PCAP-based dataset/ cache in this checkout, nothing to compare.")

### Notes and caveats

**Zeek instead of PCAP parsing.** Zeek writes non-printable bytes of a query name as `\xNN`;
`decode_query` turns them back into bytes, which reproduces the old hand-rolled parser's names on all 105
captures. Zeek sometimes logs a query and its response as two records (long-lived tunnel connections, responses
slower than 10 s, capture boundaries). They are re-joined when the addresses, ports, transaction ID and query
match within 30 s, and responses whose query isn't in the data are dropped.

**Out-of-order PCAPs.** The GraphTunnel `normal` chunks are not stored in time order. Zeek on the raw files
produced meaningless timestamps and unpaired responses, so those files are time-sorted before Zeek
(`reorder_and_rezeek.py`, see `DATA_PIPELINE.md`).

**Why per-window aggregates.** A single high-entropy query is only mildly suspicious. A tunnel shows up as many
unique names under one domain in a short time. Aggregating per 60 s window (instead of per capture) keeps the
features independent of how long a capture ran and matches what live scoring can compute from consecutive
30 s chunks.

**Why splits are by capture and time, not by row.** Rows of one session are highly correlated, so a random row
split would leak a session into its own test set. Tunnel captures are split along their timeline with a gap, and
unknownTunnel / crossEndPoint are never used for fitting or model selection.

**Artefact-suspect features.** The `normal` captures are a scripted crawl: 99% A queries for bare registered
domains, each looked up about once. Several features separate `normal` from tunnels almost perfectly for that
reason (query length, labels, per-domain volume). `domain_qtype_diversity` (normal only asks for A records) and
`no_response_ratio` (the wildcard capture left a third of its queries unanswered) look like recording artefacts
rather than tunnelling behaviour. In run 1, leaving these two out raised unseen-tool recall from 97.09% to 99.18% at 0.00% false positives on held-out normal and wildcard, so the default feature set (`zfe.FEATURE_COLUMNS`) now leaves them out. They are still computed and available as `zfe.FEATURE_SETS['all']`.
`response_latency` (resolver-dependent) and `response_size` (not available per query in Zeek) are not features.

## Part 2: model training (BiLSTM + multi-head attention)

`Build_model` and `Build_experiment` are unchanged from the PCAP-based notebook: same architecture,
hyperparameters, early stopping, learning-rate schedule and balanced class weights. What changed is the input:
Zeek features, one-hot encoding with fixed category levels, sampled train/val rows, and the evaluation and saving
around the training calls. Since run 2 the epoch cap passed to `Build_model` is 150 instead of 50 (early-stopping
patience unchanged): 40 of the 50 run-1 models, including every config A and B model, stopped at the
50-epoch cap.

In [ ]:
# Imported via tensorflow.keras instead of bare `keras`. The standalone
# `keras` pip package (Keras 3) has to version-match whatever TensorFlow
# is installed; if the two drift apart (e.g. an old conda TensorFlow next
# to a freshly pip-installed keras) names like `keras.optimizers.Adam` can
# go missing even though `keras.layers` still works. Going through
# tensorflow.keras sidesteps that -- it's guaranteed consistent with
# whatever TensorFlow version is actually installed.
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.layers import Dropout, BatchNormalization, Bidirectional, LSTM, MultiHeadAttention, Dense, Input, Concatenate, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

print("tensorflow", tf.__version__, "| keras", keras.__version__)

In [ ]:
seed = 0
tf.random.set_seed(seed)
# Also seeds Python's, NumPy's and Keras' generators so a rerun starts from the
# same state. (TensorFlow on CPU is still not guaranteed bit-for-bit identical.)
keras.utils.set_random_seed(seed)

In [ ]:
def prepare_inputs(config, feature_set=zfe.DEFAULT_FEATURE_SET, masks=None):
    # Model inputs for one run, following the original notebook's steps:
    # one-hot encode the categorical columns (fixed levels, drop_first) ->
    # StandardScaler fitted on the training rows -> reshape to
    # (rows, 1, features) -> LabelEncoder for benign/tunnel.
    masks = {**zx.split_masks(features, config), **(masks or {})}
    columns = zfe.FEATURE_SETS[feature_set]

    X_train_numeric = ma.encode_inputs(features.loc[masks['train'], columns])
    input_columns = list(X_train_numeric.columns)
    X_val_numeric = ma.encode_inputs(features.loc[masks['val'], columns], input_columns)
    X_test_numeric = ma.encode_inputs(features.loc[masks['test'], columns], input_columns)
    X_eval_numeric = ma.encode_inputs(features.loc[masks['evaluation'], columns], input_columns)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_numeric)

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(features.loc[masks['train'], 'Label'])

    eval_meta = zx.evaluation_meta(features, config, masks['evaluation'])
    return {
        'X_train': ma.to_tensor(X_train_scaled),
        'X_val': ma.to_tensor(scaler.transform(X_val_numeric)),
        'X_test': ma.to_tensor(scaler.transform(X_test_numeric)),
        'X_eval': ma.to_tensor(scaler.transform(X_eval_numeric)),
        'y_train': y_train_encoded,
        'y_val': label_encoder.transform(features.loc[masks['val'], 'Label']),
        'y_test': label_encoder.transform(features.loc[masks['test'], 'Label']),
        'eval_meta': eval_meta,
        'scaler': scaler,
        'label_encoder': label_encoder,
        'input_columns': input_columns,
        'train_rows': features.loc[masks['train'], 'Label'].value_counts().to_dict(),
        'val_rows': features.loc[masks['val'], 'Label'].value_counts().to_dict(),
    }

In [ ]:
def Build_model(X_train, y_train, X_val, y_val, X_test, y_test, optimizer,
                n_of_hidden_layers, n_neurons, activation='tanh', dropout_rate=0.3, l2_reg=1e-4,
                epochs=50, batch_size=128, early_stopping=True, verbose=1):
    inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))
    x = inputs
    
    # Adding Bidirectional LSTM layers with Dropout and BatchNormalization.
    # kernel/recurrent L2 added: with only 1 timestep of input, this stack was
    # memorizing the training set (loss -> ~0 within 1 epoch) instead of
    # learning a boundary that holds up on unseen sessions -- L2 plus the
    # smaller n_neurons/n_of_hidden_layers passed in below are what actually
    # rein that in.
    for _ in range(n_of_hidden_layers):
        x = Bidirectional(LSTM(n_neurons, activation=activation, return_sequences=True,
                                kernel_regularizer=l2(l2_reg), recurrent_regularizer=l2(l2_reg)))(x)
        x = Dropout(dropout_rate)(x)
        x = BatchNormalization()(x)
    
    attention = MultiHeadAttention(num_heads=4, key_dim=n_neurons)(x, x)
    x = Concatenate()([x, attention])
    
    x = Flatten()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    
    # Inferred from the data instead of hardcoded: the original 6-class
    # NSL-KDD-style labels became binary benign/tunnel DNS labels.
    n_classes = len(np.unique(y_train))
    outputs = Dense(n_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    es = EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="auto",
        baseline=None,
        restore_best_weights=True,
    )

    # ReduceLROnPlateau: about 1 in 5 runs of this architecture was collapsing
    # to always predicting the majority class and getting stuck there for
    # every remaining epoch (val_loss flat/rising, val_accuracy pinned at the
    # majority-class rate) -- giving a stuck run a shrinking learning rate
    # instead of only a fixed one gives it a chance to escape that plateau
    # rather than early-stopping straight out of it.
    rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=0)

    # class_weight: train is ~1.22:1 benign:tunnel and val ~2.6:1 now that
    # wildcard hard negatives are in both -- balancing the loss keeps that
    # mild imbalance from being what tips an unlucky init toward the
    # always-predict-benign shortcut above.
    classes = np.unique(y_train)
    class_weight = dict(zip(classes, compute_class_weight('balanced', classes=classes, y=y_train)))

    if early_stopping:
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, callbacks=[es, rlrop], class_weight=class_weight, verbose=verbose)
    else:
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, batch_size=batch_size, callbacks=[rlrop], class_weight=class_weight, verbose=verbose)
    
    train_evaluation = model.evaluate(X_train, y_train, verbose=0)
    test_evaluation = model.evaluate(X_test, y_test, verbose=0)
    validation_evaluation = model.evaluate(X_val, y_val, verbose=0)
    
    return model, history, train_evaluation, validation_evaluation, test_evaluation

In [ ]:
def Build_experiment(X_train, y_train, X_val, y_val, X_test, y_test,
                     n_of_hidden_layers, n_neurons, activation='tanh', dropout_rate=0.3, l2_reg=1e-4, epochs=50, batch_size=128, n_of_models=5, early_stopping=True, verbose=1):
    models_dict = {'models': [], 'history': []}

    models_train_acc = []
    models_test_acc = []
    models_valid_acc = []

    for j in range(n_of_models):
        # Recreate optimizer for each model
        optimizer = Adam(learning_rate=0.001)
        
        # Build model
        model, history, train_evaluation, valid_evaluation, test_evaluation = Build_model(
            X_train, y_train, X_val, y_val, X_test, y_test, optimizer,
            n_of_hidden_layers, n_neurons, activation=activation, dropout_rate=dropout_rate, l2_reg=l2_reg, epochs=epochs, batch_size=batch_size, early_stopping=early_stopping, verbose=verbose
        )
        
        # Save data
        models_train_acc.append(train_evaluation[1])
        models_test_acc.append(test_evaluation[1])
        models_valid_acc.append(valid_evaluation[1])

        models_dict['models'].append(model)
        models_dict['history'].append(history)

    accuracies_dict = {
        "Min_train_acc": min(models_train_acc),
        "Max_train_acc": max(models_train_acc),
        "AVG_train_acc": np.mean(models_train_acc),
        "Min_test_acc": min(models_test_acc),
        "Max_test_acc": max(models_test_acc),
        "AVG_test_acc": np.mean(models_test_acc),
        "Min_valid_acc": min(models_valid_acc),
        "Max_valid_acc": max(models_valid_acc),
        "AVG_valid_acc": np.mean(models_valid_acc)
    }

    return accuracies_dict, models_dict, models_train_acc, models_test_acc, models_valid_acc

### Running and scoring one configuration

`run_configuration` trains `N_OF_MODELS` models with `Build_experiment` and the hyperparameters of the
PCAP-based notebook (epoch cap raised to 150), scores every model on all test and held-out rows, and saves:

* the models, scaler, label encoder and `features.json` to `models/zeek_bilstm/<RUN_NAME>/<name>/` (gitignored),
* the metrics and per-epoch loss histories to `results/runs/<RUN_NAME>/<name>.json` (committed).

Each configuration reseeds before training, so its result doesn't depend on which configurations ran before it.

In [ ]:
import hashlib

HYPERPARAMETERS = dict(n_of_hidden_layers=1, n_neurons=24, activation='tanh', dropout_rate=0.5,
                       l2_reg=1e-4, epochs=150, batch_size=128, early_stopping=True)
SPLITS_SHA256 = hashlib.sha256(ds.SPLITS_PATH.read_bytes().replace(b'\r\n', b'\n')).hexdigest()
RUN_MODELS_DIR = ma.MODELS_DIR / RUN_NAME
results = {}


def run_configuration(name, config, feature_set=zfe.DEFAULT_FEATURE_SET, masks=None, relabel=None,
                      description=None, extra=None):
    t0 = time.time()
    # Same starting point for every configuration, whatever ran before it in this kernel.
    keras.utils.set_random_seed(seed)
    data = prepare_inputs(config, feature_set, masks)
    if relabel is not None:
        data['eval_meta'] = relabel(data['eval_meta'])
    accuracies, models_dict, train_acc, test_acc, valid_acc = Build_experiment(
        data['X_train'], data['y_train'], data['X_val'], data['y_val'], data['X_test'], data['y_test'],
        n_of_models=N_OF_MODELS, verbose=0, **HYPERPARAMETERS)

    per_run, window_sizes = zx.score_models(models_dict['models'], data['X_eval'], data['eval_meta'],
                                            data['label_encoder'])
    histories = [{key: [float(v) for v in values] for key, values in h.history.items()}
                 for h in models_dict['history']]
    result = {
        'name': name, 'config': config, 'feature_set': feature_set, 'accuracies': accuracies,
        'models': models_dict, 'summary': zx.summarise(per_run), 'per_run': per_run,
        'window_sizes': window_sizes, 'epochs': [len(h['loss']) for h in histories], 'histories': histories,
        'input_columns': data['input_columns'], 'train_rows': data['train_rows'], 'val_rows': data['val_rows'],
        'label_encoder': data['label_encoder'], 'test_inputs': (data['X_test'], data['y_test']),
        'seconds': time.time() - t0,
        'settings': {'epoch_cap': HYPERPARAMETERS['epochs'], 'hyperparameters': HYPERPARAMETERS,
                     'row_caps': ds.ROW_CAPS, 'sampling_seed': ds.SAMPLING_SEED, 'model_seed': seed,
                     'window_seconds': zfe.WINDOW_SECONDS, 'feature_columns': zfe.FEATURE_SETS[feature_set],
                     'splits_sha256': SPLITS_SHA256},
        'git': ma.git_info(), 'environment': ma.environment_info(),
        'trained_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'source': 'trained by dns_tunneling_bilstm_model.ipynb',
        **(extra or {}),
    }
    ma.save_artifacts(RUN_MODELS_DIR / name, models_dict['models'], data['scaler'], data['label_encoder'],
                      zfe.FEATURE_SETS[feature_set], data['input_columns'], {
                          'run': RUN_NAME,
                          'name': name,
                          'config': config,
                          'config_description': description or ds.CONFIG_DESCRIPTIONS.get(config),
                          'feature_set': feature_set,
                          'row_caps': ds.ROW_CAPS,
                          'sampling_seed': ds.SAMPLING_SEED,
                          'splits_file': 'Data/processed/GraphTunnel/splits.csv',
                          'splits_sha256': SPLITS_SHA256,
                          'training_rows': data['train_rows'],
                          'validation_rows': data['val_rows'],
                          'n_of_models': N_OF_MODELS,
                          'hyperparameters': HYPERPARAMETERS,
                          'class_weight': 'balanced (computed in Build_model)',
                          'epochs_trained': result['epochs'],
                          'best_epochs': [int(np.argmin(h['val_loss'])) + 1 for h in histories],
                          'evaluation_avg': result['summary']['avg'].to_dict(),
                          'not_for_evaluation': False,
                      })
    zx.save_result(RUN_NAME, name, result)
    print(f"{name}: {result['seconds'] / 60:.1f} min, epochs {result['epochs']}, "
          f"train rows {data['train_rows']}, input columns {len(data['input_columns'])}")
    return result


def spread_table(names, metrics=None):
    # avg (min - max) per metric, as percentages, one column per run name.
    table = {}
    for name in names:
        s = results[name]['summary']
        table[name] = s.apply(lambda r: f"{100 * r['avg']:.2f}% ({100 * r['min']:.2f}-{100 * r['max']:.2f})", axis=1)
    table = pd.DataFrame(table)
    return table if metrics is None else table.reindex([m for m in metrics if m in table.index])

## Configs B (primary) and A (stress test)

* **B:** wildcard hard negatives in train and val; wildcard 00007–00012 held out.
* **A:** no wildcard in train or val; all 13 wildcard captures held out.

`RUN_CONFIGS` in the first cell selects which of them this run trains.

In [ ]:
for config in RUN_CONFIGS:
    results[config] = run_configuration(config, config)

if 'B' in results:
    # Names the plotting and printout cells below use (config B).
    lstm_models_dict = results['B']['models']
    lstm_accuracies_dict = results['B']['accuracies']
    X_test_scaled_tensor, y_test_encoded = results['B']['test_inputs']
    label_encoder = results['B']['label_encoder']

In [ ]:
def plot_history(dict_of_lists, type='loss'):
    axis_idx = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    for i in range(len(dict_of_lists)):
        ax = axes[axis_idx[i][0]][axis_idx[i][1]]
        # Summarize history for accuracy
        ax.plot(dict_of_lists[i].history[type])
        ax.plot(dict_of_lists[i].history[f'val_{type}'])
        ax.set_title(f'Model {i+1} {type.title()}', size=15)
        ax.set_ylabel(f'{type.title()}', size=10)
        ax.set_xlabel('Epoch', size=10)
        ax.legend(['train', 'test'], loc='upper left')
    plt.suptitle(f'Models {type.title()} per Epoch', size=15, y=0.93)
    plt.show()

if 'B' in results:
    plot_history(lstm_models_dict['history'], type='accuracy')
    plot_history(lstm_models_dict['history'], type='loss')

In [ ]:
if 'B' in results:
    # Display the average confusion matrix (config B, in-distribution test)
    n_classes = len(label_encoder.classes_)  # was hardcoded to 6 for the old multi-class labels
    cm = np.zeros(shape=(n_classes, n_classes))
    for model in lstm_models_dict['models']:
        pred = model.predict(X_test_scaled_tensor, batch_size=4096, verbose=0).argmax(axis=1)
        cm += confusion_matrix(y_test_encoded, pred, labels=range(n_classes))
    cm_avg = cm / len(lstm_models_dict['models'])

    # Set custom color palette
    colors = ["#FF0B04", "#4374B3"]
    sns.set_palette(sns.color_palette(colors))
    sns.set(font_scale=1.5)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm_avg, display_labels=label_encoder.classes_)
    disp = disp.plot(cmap=plt.cm.Blues, values_format='.0f')

    fig = disp.ax_.get_figure()
    fig.set_figwidth(10)
    fig.set_figheight(10)

    plt.title('Average confusion matrix, config B in-distribution test')
    plt.grid(False)
    plt.show()
    sns.set(font_scale=1.0)

In [ ]:
# Min/Max alongside Average: the average alone hides a real failure mode
# here -- a collapsed single run (always predicting the majority class) pulls
# the average down while looking, at a glance, like "slightly lower accuracy"
# rather than what it actually is (4 good runs + 1 broken one). Min close to
# Max is the signal that all n_of_models runs actually converged properly.
if 'B' in results:
    for split in ["train", "valid", "test"]:
        print(f"{split.title()} Accuracy (config B) -- "
              f"min={lstm_accuracies_dict.get(f'Min_{split}_acc', float('nan')):.4f}  "
              f"max={lstm_accuracies_dict.get(f'Max_{split}_acc', float('nan')):.4f}  "
              f"avg={lstm_accuracies_dict.get(f'AVG_{split}_acc', float('nan')):.4f}")

## Held-out results, configs B and A side by side

Rates are the share of rows classified as tunnel: false positive rates for benign rows, recall for tunnel rows.
Each cell is the average over the runs with min–max in brackets. `fpr_wildcard_00007_00012` compares the two
configurations on the same wildcard captures; in config A, `fpr_wildcard` covers all 13.

In [ ]:
main_names = [name for name in ('B', 'A') if name in results]
main_metrics = ['test_accuracy', 'test_benign', 'test_tunnel', 'fpr_normal', 'fpr_wildcard_00007_00012',
                'fpr_wildcard', 'unseen_tool', 'unseen_platform', 'collapsed']
if main_names:
    print(spread_table(main_names, main_metrics).to_string())
    per_tool = [m for m in results[main_names[0]]['summary'].index if m.startswith(('unseen_tool/', 'unseen_platform/'))]
    print()
    print(spread_table(main_names, per_tool).to_string())
for name in main_names:
    sizes = results[name]['window_sizes']
    print(f"\nConfig {name}, rate by queries in the row's window (rows in brackets):")
    print(sizes.assign(cell=sizes.apply(lambda r: f"{100 * r['rate']:.2f}% ({r['rows']:,})" if r['rows'] else '-', axis=1))
          .pivot(index='group', columns='bucket', values='cell')[['<10', '10-99', '>=100']].to_string())

## Feature-set ablations (config B)

Same rows, splits and training setup; only the input columns change. Runs when `RUN_ABLATIONS` is True, with every
feature set other than the default (`all` is run 1's default, with the artefact-suspect pair).

In [ ]:
if RUN_ABLATIONS:
    for feature_set in zfe.FEATURE_SETS:
        if feature_set != zfe.DEFAULT_FEATURE_SET:
            results[f'B-{feature_set}'] = run_configuration(f'B-{feature_set}', 'B', feature_set)
    ablation_names = [n for n in ['B'] + [f'B-{fs}' for fs in zfe.FEATURE_SETS] if n in results]
    print(spread_table(ablation_names, ['test_accuracy', 'fpr_normal', 'fpr_wildcard', 'unseen_tool',
                                        'unseen_platform', 'collapsed']).to_string())

## Leave-one-tunnel-family-out cross-validation

This replaces the earlier row-level `KFold` cross-validation, which shuffled rows of the same session into
different folds and contradicted the capture-level split rule. Each fold uses config B's benign data, trains on
the train segments of four tunnel families (validating on their val segments), and is scored on every row of the
fifth family's captures plus the held-out normal and wildcard captures.

In [ ]:
lofo = {}
for family in RUN_LOFO_FAMILIES:
    fold_masks, relabel = zx.lofo_masks(features, family, ds.PRIMARY_CONFIG)
    name = f'lofo-{family}'
    lofo[family] = results[name] = run_configuration(
        name, ds.PRIMARY_CONFIG, masks=fold_masks, relabel=relabel,
        description=f'leave-one-family-out: {family} held out, config {ds.PRIMARY_CONFIG} benign data',
        extra={'family_rows': int(fold_masks['test'].sum())})

if lofo:
    print(pd.DataFrame({family: r['summary']['avg'] for family, r in lofo.items()})
          .reindex(['held_out_family', 'fpr_normal', 'fpr_wildcard', 'collapsed']).map(lambda v: f'{100 * v:.2f}%').to_string())

In [ ]:
if lofo:
    fig, ax = plt.subplots(figsize=(10, 4))
    families = list(lofo)
    recall = [lofo[f]['summary'].loc['held_out_family'] for f in families]
    avg = np.array([r['avg'] for r in recall]) * 100
    low = avg - np.array([r['min'] for r in recall]) * 100
    high = np.array([r['max'] for r in recall]) * 100 - avg
    ax.bar(families, avg, yerr=[low, high], capsize=6, color='#4374B3')
    ax.set_ylim(0, 105)
    ax.set_ylabel('Recall on the held-out family (%)')
    ax.set_title(f'Leave-one-family-out: recall (avg, min-max over {N_OF_MODELS} runs)')
    plt.show()

## Write this run's section of `results/zeek_run.md`

The section is rendered from `results/runs/<RUN_NAME>/*.json`, so it covers every configuration of the run even when
the run was split across several kernels, and it replaces only its own section of the file. Loss curves go to
`results/figures/`.

In [ ]:
RUN_DESCRIPTION = [
    "Two changes from run 1, and a targeted re-run (config B and the leave-one-family-out folds; config A, the "
    "ablations and the final model are not retrained yet):",
    "",
    "1. **Default feature set: `all_minus_artefact_suspect`** (28 features; `domain_qtype_diversity` and "
    "`no_response_ratio` left out). In run 1's config B ablation, dropping them raised recall on unseen tools from "
    "97.09% (97.07–97.13) to 99.18% (99.11–99.22) and on the unseen platform from 99.39% to 99.60%, with false "
    "positive rates on held-out normal 0.01% → 0.00% and held-out wildcard 0.00% → 0.00%. Both features are still "
    "computed and form part of the `all` feature set.",
    "2. **Epoch cap 50 → 150**, early-stopping patience and everything else in `Build_model` unchanged: 40 of the "
    "50 run-1 models hit the cap, including all 10 config A and B models (early stopping fired only in some "
    "ablation and dnscat2-fold models). Run 1's config B loss curves, from its notebook output: "
    "![run 1 config B loss](figures/run1_all_50ep_B_loss.png)",
    "",
    "Each configuration now reseeds (seed 0) before training. For config B the run-1 ablation column (same 28 "
    "features, 50 epochs) isolates the effect of the longer training; the leave-one-family-out comparison changes "
    "both at once. The run-1 columns were re-scored from the saved run-1 models and match the tables above.",
]
if RENDER_REPORT:
    report = zx.write_run_section(RUN_NAME, RUN_TITLE, description=RUN_DESCRIPTION, reference_name=REFERENCE_RUN,
                                  run_label='run 2', reference_label='run 1',
                                  extra_reference={'B': ['B-all_minus_artefact_suspect']})
    print(f'Wrote {report}')